In [ ]:
%load_ext autoreload
%autoreload 2

import utils
import importlib

importlib.reload(utils)

In [ ]:
from pathlib import Path

THIS_DIR = Path.cwd()
RAW_DIR = THIS_DIR / "generated" / "raw"

VCF_NAME = (
    "1kGP_high_coverage_Illumina.chr22."
    "filtered.SNV_INDEL_SV_phased_panel.vcf.gz"
)
PANEL_NAME = "integrated_call_samples_v3.20130502.ALL.panel"
GTF_NAME = "gencode.v50.annotation.gtf.gz"

SOURCES = {
    VCF_NAME: (
        "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/"
        "data_collections/1000G_2504_high_coverage/working/"
        "20220422_3202_phased_SNV_INDEL_SV/"
        f"{VCF_NAME}"
    ),
    PANEL_NAME: (
        "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/"
        f"release/20130502/{PANEL_NAME}"
    ),
    GTF_NAME: (
        "https://ftp.ebi.ac.uk/pub/databases/gencode/"
        f"Gencode_human/release_50/{GTF_NAME}"
    ),
}

print("current working directory:", THIS_DIR)

In [ ]:
for filename, url in SOURCES.items():
    utils.download_if_missing(url, RAW_DIR / filename)

# check that all files exist
for filename in SOURCES:
    assert (RAW_DIR / filename).exists(), f"File {filename} not found in {RAW_DIR}"
print("All files downloaded successfully.")

In [ ]:
GENOTYPE_DIR = THIS_DIR / "generated" / "genotype"
WORK_DIR = THIS_DIR / "generated" / "work"
PGEN_PREFIX = GENOTYPE_DIR / "chr22"

utils.create_pgen(
    vcf_path=RAW_DIR / VCF_NAME,
    panel_path=RAW_DIR / PANEL_NAME,
    out_prefix=PGEN_PREFIX,
    keep_path=WORK_DIR / "phase3.keep",
)

In [ ]:
GENE_PANEL_PATH = THIS_DIR / "generated" / "gene_panel.tsv"
ANNOTATION_PATH = THIS_DIR / "generated" / "annotation.tsv"

utils.create_inputs_from_gencode(
    gtf_path=RAW_DIR / GTF_NAME,
    pvar_path=PGEN_PREFIX.with_suffix(".pvar"),
    gene_panel_path=GENE_PANEL_PATH,
    annotation_path=ANNOTATION_PATH,
    chr="22",
)

In [ ]:
COVARIATES_PATH = THIS_DIR / "generated" / "covariates.tsv"

utils.create_covariates(
    panel_path=RAW_DIR / PANEL_NAME,
    psam_path=PGEN_PREFIX.with_suffix(".psam"),
    out_path=COVARIATES_PATH,
)

In [ ]:
PHENOTYPE_PATH = THIS_DIR / "generated" / "phenotype.csv"

utils.create_phenotype(
    psam_path=PGEN_PREFIX.with_suffix(".psam"),
    out_path=PHENOTYPE_PATH,
    seed=260323,
)

In [ ]:
import os
import shutil

os.environ["PATH"] = (
    "/usr/local/bin"
    + os.pathsep
    + os.environ.get("PATH", "")
)

print(shutil.which("plink2"))

In [ ]:
import sys

REPO_ROOT = next(
    directory
    for directory in (THIS_DIR, *THIS_DIR.parents)
    if (directory / "rewrite" / "preprocessing").is_dir()
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
from rewrite.preprocessing.input import load_inputs
from rewrite.preprocessing.model import PrepOptions
from rewrite.preprocessing.pipeline import prepare_blocks

GENERATED_DIR = THIS_DIR / "generated"
PGEN_PREFIX = GENERATED_DIR / "genotype" / "chr22"

inputs = load_inputs(
    pgen_prefix=PGEN_PREFIX,
    gene_panel_path=GENERATED_DIR / "gene_panel.tsv",
    annotation_path=GENERATED_DIR / "annotation.tsv",
    phenotype_path=GENERATED_DIR / "phenotype.csv",
    covariate_path=GENERATED_DIR / "covariates.tsv",
)

# Several genes, not one: gene order, empty genes and variants shared between
# overlapping genes only show up with more than one block.
SMOKE_GENE_IDS = tuple(gene.gene_id for gene in inputs.gene_panel[:15])
SMOKE_OUT_DIR = GENERATED_DIR / "preprocessed" / "smoke_multi_gene"

options = PrepOptions(
    chromosome="22",
    gene_selection=SMOKE_GENE_IDS,
    # The generated annotation sets LoF=HC on every row, so masking on LoF alone
    # filters nothing. consequence is what actually cuts the variant list.
    mask={
        "LoF": "HC",
        "consequence": {"missense_variant"},
    },
    phenotype_columns=("phenotype",),
    covariate_columns=(
        "superpop_AFR",
        "superpop_AMR",
        "superpop_EAS",
        "superpop_EUR",
    ),
    samples_per_cohort=500,
    sample_seed=42,
    role_seed=42,
    out_dir=SMOKE_OUT_DIR,
)

result = prepare_blocks(
    inputs=inputs,
    options=options,
)

print(f"created: {result}")


In [ ]:
import numpy as np

# Structural invariants of the written output. These are cheap and they are what
# the secure step relies on; the dosage-level check lives in
# rewrite/preprocessing/test_end_to_end.py.
gene_ids = (result / "genes.txt").read_text().split()
block_sizes = [int(size) for size in (result / "block_sizes.txt").read_text().split()]
n_a = len((result / "A" / "cov.txt").read_text().splitlines())
n_b = len((result / "B" / "cov.txt").read_text().splitlines())

assert len(gene_ids) == len(block_sizes) == len(SMOKE_GENE_IDS)
assert n_a == n_b == options.samples_per_cohort
assert len((result / "pos.txt").read_text().splitlines()) == sum(block_sizes)
assert len((result / "A" / "pheno.txt").read_text().splitlines()) == n_a
assert len((result / "B" / "pheno.txt").read_text().splitlines()) == n_b

for index, m_public in enumerate(block_sizes):
    public_a = np.fromfile(result / "A" / "geno" / f"block.{index}.bin", dtype=np.int8)
    public_b = np.fromfile(result / "B" / "geno" / f"block.{index}.bin", dtype=np.int8)
    private_b = np.fromfile(result / "B" / "private" / f"block.{index}.bin", dtype=np.int8)

    assert public_a.size == n_a * m_public, index
    assert public_b.size == n_b * m_public, index
    assert private_b.size % n_b == 0, index
    for block in (public_a, public_b, private_b):
        assert block.size == 0 or set(np.unique(block)) <= {0, 1, 2}, index

m_private = [
    np.fromfile(result / "B" / "private" / f"block.{index}.bin", dtype=np.int8).size // n_b
    for index in range(len(block_sizes))
]
print(f"n_a={n_a} n_b={n_b}")
print(f"public per gene:  {block_sizes}")
print(f"private per gene: {m_private}")
print(f"empty public blocks: {block_sizes.count(0)}")
print("all invariants hold")
